# Xu Lab BindCraft2 on CMRI HPC - binder discovery platform

A general-purpose binder design platform: pick any protein target, pick any binder format
BindCraft2 supports, submit the campaign to an H200 through SLURM, and read the ranked results
back. Adapted from the official `bindcraft.ipynb` for **regular Jupyter on the CMRI HPC**; the
upstream notebook is untouched.

| Upstream Colab notebook | This notebook |
| --- | --- |
| mounts Google Drive | runs from a source tree in `$HOME` and writes to a project-storage tree |
| `pip install`s into the runtime | uses the existing `bindcraft2` conda environment; installs nothing |
| `google.colab.output.enable_custom_widget_manager()` | plain `ipywidgets`, no Colab imports |
| runs the campaign **inside** the notebook process | submits a **SLURM batch job** to an H200 |
| one hardcoded example target | search RCSB by name or UniProt, or upload / paste / point at a file |
| `files.download(...)` | writes an export folder plus a zip, with Jupyter file links |
| downloads AlphaFold weights on demand | reuses the parameter set already on the cluster |

**The defaults are an example, not a restriction.** Step 2 opens on **human CD19** (RCSB 6AL5)
and step 3 on the **scFv** format, because that is the campaign this platform was first driven
with end to end. Every part of both steps is a control: any RCSB entry, any uploaded or pasted
target, any of the ten binder formats.

**What this notebook does not do:** it never designs anything in the kernel. Every campaign is a
batch job, so you can close the notebook, log out, and come back to the same campaign. The
kernel also never claims GPU success - the H200 is only ever evidenced by the job's own log, which
the launcher writes and `validate_outputs.py` reads.

**Run order:** cell 1 once per session, then 2-4 to submit, then 5-9 for any campaign (past or
present). Cells 5-9 work on a campaign that is still running.

**Filters stay on.** The Colab demo's `no_filters` shortcut is deliberately not offered here: a
campaign without acceptance filters produces numbers that mean nothing.


## 1 - Set up and check the environment

In [ ]:
# Imports the helper layer beside this notebook and reports what is actually present.
# Nothing is installed and no GPU claim is made: a Jupyter node is not the batch allocation.
import os, sys, json, importlib
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
for candidate in (NOTEBOOK_DIR, NOTEBOOK_DIR / "lab" / "notebooks",
                  Path(os.environ.get("XULAB_BC2_HOME", "")) / "lab" / "notebooks",
                  Path("/home/ypan/BindCraft2/BindCraft2_deploy/lab/notebooks")):
    if (candidate / "xulab_hpc.py").is_file():
        sys.path.insert(0, str(candidate))
        break

import xulab_hpc as hpc
importlib.reload(hpc)

# Which deployment this kernel is driving. Every path below comes from the XULAB_*
# environment variables, so a development deployment is selected by sourcing its
# env file before starting Jupyter - never by editing a tracked file.
print("=" * 78)
print(f"deployment    : {hpc.DEPLOYMENT}")
print(f"source tree   : {hpc.SOURCE_TREE}")
print(f"project root  : {hpc.PROJECT_ROOT}")
print("=" * 78)
print(f"results root  : {hpc.RESULTS_ROOT}")
print(f"targets root  : {hpc.TARGETS_ROOT}")
print(f"logs root     : {hpc.LOGS_ROOT}")
print(f"job registry  : {hpc.JOB_REGISTRY}")
print(f"interpreter   : {hpc.CONDA_PYTHON}")
print(f"launcher      : {hpc.SLURM_SCRIPT}")
print()

ENVIRONMENT = hpc.environment_report()
width = max(len(name) for name, _, _ in ENVIRONMENT)
for name, status, detail in ENVIRONMENT:
    print(f"{status:11s} {name:<{width}}  {detail}")

status, detail = hpc.imported_bindcraft_path()
print(f"\n{status:11s} bindcraft import origin  {detail}")

BLOCKING = [name for name, state, _ in ENVIRONMENT if state == "FAIL"]
if BLOCKING or status == "FAIL":
    print("\nBLOCKED: fix the FAIL lines before submitting. Missing directories are created with")
    print(f"  mkdir -p {hpc.PROJECT_ROOT}/{{results,logs,cache/jax,tmp,targets}}")
    print("and the full checklist is lab/scripts/hpc_preflight.sh on a login node.")
else:
    print("\nREADY to configure a campaign (GPU use is still unproven until a job runs).")


## 2 - Target

Six ways to name a target, in one form. Whichever you use, the result is the same: a target file
on the cluster under the **targets root** printed in cell 1, plus a settings entry this notebook
writes into the campaign JSON for you. Nothing here is edited by hand.

| Source | What it needs | Notes |
| --- | --- | --- |
| **search RCSB** | a protein name, a UniProt accession or a PDB ID | needs internet access from this kernel |
| **a PDB ID** | e.g. `6AL5` | downloads the entry, then pick the chain |
| **upload a file** | `.pdb`, `.cif` or `.fasta` from your machine | stored unchanged under `targets/uploads/` |
| **paste a sequence** | FASTA or a bare sequence | written as a FASTA target |
| **a file on the cluster** | an absolute path | used where it is; nothing is copied |
| **a shipped preset** | one of the targets BindCraft2 ships | upstream's own presets, unchanged |

**Chain selection is never done for you.** When an entry holds more than one polymer chain - and
an antigen–antibody complex always does - the form lists the chains with their sizes and the
entity description RCSB gives for each, and waits. The longest chain is usually the antibody, not
the antigen, so guessing would be worse than asking.

**What gets reported before you commit:** the entry title, method and resolution; each entity's
description, author chain, length, organism and UniProt accession; the residue range actually
present; the ordered segments and the gaps; the engineered mutations and expression tags the file
itself declares; and how the entity's numbering maps to its reference sequence. The target file is
written with the **deposited author numbering unchanged**, because that is the numbering hotspots
have to be given in.

**Hotspots and coldspots are optional.** Leave them empty and BindCraft2 picks its own interface.
Give them to steer where the binder lands, in this target file's own numbering.

**On accessibility:** the geometry report measures the site's span, its distance to each end of
the modelled chain, and the nearest glycosylation sequon. It does not tell you whether the site is
reachable on a cell - membrane proximity, partner-protein occlusion and glycan shielding are not
properties of a soluble fragment's coordinates. The report says so rather than implying otherwise.

If RCSB is unreachable from this kernel, the search and PDB-ID routes say so and name the
alternative; prepare the file on a machine with internet access with
`python lab/scripts/prepare_target.py --entry <id> --chain <letter>` and copy it across, or use
the upload / paste / cluster-path routes.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

STYLE = {"description_width": "180px"}
def wide(width="560px"):
    return widgets.Layout(width=width)   # a Layout belongs to one widget, so build a new one

TARGETS = []          # the campaign's target entries, in the order added
LOCATED = {}          # the file the current source resolved to, plus its description

source = widgets.Dropdown(
    options=["search RCSB by name / UniProt / PDB ID", "a PDB ID", "upload a file",
             "paste a sequence", "a file on the cluster", "a shipped preset"],
    value="search RCSB by name / UniProt / PDB ID",
    description="target from", style=STYLE, layout=wide("640px"))

query = widgets.Text(value="CD19", description="search", placeholder="CD19, P15391 or 6AL5",
                     style=STYLE, layout=wide("640px"))
run_search = widgets.Button(description="search RCSB", layout=widgets.Layout(width="160px"))
hits = widgets.Dropdown(options=[], description="entries found", style=STYLE, layout=wide("640px"))
pdb_id = widgets.Text(value="6AL5", description="PDB ID", placeholder="6AL5",
                      style=STYLE, layout=wide("340px"))
inspect = widgets.Button(description="fetch and inspect", button_style="info",
                         layout=widgets.Layout(width="190px"))
upload = widgets.FileUpload(accept=".pdb,.ent,.cif,.mmcif,.fasta,.fa,.faa,.seq,.txt",
                            multiple=False, description="choose a file",
                            layout=widgets.Layout(width="260px"))
read_upload = widgets.Button(description="read the upload", button_style="info",
                             layout=widgets.Layout(width="190px"))
pasted = widgets.Textarea(placeholder=">my_target\nMKTAYIAKQRQISFVKSHFSRQ...",
                          description="sequence", style=STYLE,
                          layout=widgets.Layout(width="640px", height="120px"))
file_path = widgets.Text(description="file path", placeholder=str(hpc.TARGETS_ROOT / "my_target.pdb"),
                         style=STYLE, layout=wide("640px"))
read_path = widgets.Button(description="read this file", button_style="info",
                           layout=widgets.Layout(width="190px"))
shipped = widgets.Dropdown(options=hpc.available_targets() or ["(none shipped)"],
                           value="hPDL1" if "hPDL1" in hpc.available_targets() else None,
                           description="which preset", style=STYLE, layout=wide("640px"))

chain = widgets.Dropdown(options=[], description="antigen chain", style=STYLE, layout=wide("640px"))
name = widgets.Text(value="", description="target name", placeholder="defaults to <file>_<chain>",
                    style=STYLE, layout=wide("440px"))
first_residue = widgets.Text(description="keep from residue", placeholder="empty = the first present",
                             style=STYLE, layout=wide("440px"))
last_residue = widgets.Text(description="keep to residue",
                            placeholder="empty = the last present; use it to cut a C-terminal tag",
                            style=STYLE, layout=wide("440px"))
hotspots = widgets.Text(description="hotspots (optional)", placeholder="A162,A219,A220,A222",
                        style=STYLE, layout=wide("640px"))
coldspots = widgets.Text(description="coldspots (optional)", placeholder="A90-95",
                         style=STYLE, layout=wide("640px"))
geometry = widgets.Checkbox(value=True, description="report the geometry of the hotspots",
                            indent=False)
add = widgets.Button(description="prepare and add", button_style="primary",
                     layout=widgets.Layout(width="190px"))
clear = widgets.Button(description="remove all", layout=widgets.Layout(width="140px"))
lookup_out = widgets.Output()
chosen = widgets.Output()

RCSB_SOURCES = ("search RCSB by name / UniProt / PDB ID", "a PDB ID")
FILE_SOURCES = ("upload a file", "a file on the cluster") + RCSB_SOURCES


def on_source(change=None):
    showing = {
        "search RCSB by name / UniProt / PDB ID": (query, run_search, hits),
        "a PDB ID": (pdb_id, inspect),
        "upload a file": (upload, read_upload),
        "paste a sequence": (pasted,),
        "a file on the cluster": (file_path, read_path),
        "a shipped preset": (shipped,),
    }[source.value]
    for box in (query, run_search, hits, pdb_id, inspect, upload, read_upload, pasted,
                file_path, read_path, shipped):
        box.layout.display = None if box in showing else "none"
    coordinates = source.value in FILE_SOURCES
    for box in (chain, first_residue, last_residue, geometry):
        box.layout.display = None if coordinates else "none"
    LOCATED.clear()
    chain.options = []


def _describe(path, note=""):
    """Print what a structure file holds and populate the chain dropdown from it."""
    described = hpc.describe_structure(path)
    LOCATED["path"] = str(path)
    LOCATED["described"] = described
    if note:
        print(note)
    print(f"file     : {path}")
    print(f"format   : {described['format'] or 'unrecognised'}")
    for problem in described["problems"]:
        print(f"problem  : {problem}")
    if described["format"] == "fasta":
        print(described["chain_note"])
        for record in described["chains"]:
            print(f"  {record['chain']}  {record['residue_count']} residues")
        chain.options = []
        return
    entry = LOCATED.get("entry")
    print("\nchains in the coordinates (pick the antigen, not its partner):")
    labels = []
    for item in described["chains"]:
        if not item.get("polymer"):
            print(f"  {item['chain']}  heteroatoms only")
            continue
        entity = hpc.entity_of_chain(entry, item["chain"]) if entry else {}
        description = entity.get("description") or ""
        organism = "; ".join(entity.get("organisms") or [])
        accession = "/".join(entity.get("uniprot") or [])
        print(f"  {item['chain']}  {item['residue_count']:>4} residues  "
              f"{item['first']}-{item['last']}  {description}"
              + (f"  [{accession}]" if accession else "")
              + (f"  ({organism})" if organism else ""))
        print(f"       segments : {hpc.span_text(item['segments'])}")
        if item["gaps"]:
            print(f"       absent   : {hpc.span_text(item['gaps'])}  "
                  "(not modelled by the depositors; left absent, never filled in)")
        if item["hetero"]:
            print("       hetero   : " + ", ".join(f"{label} x{count}" for label, count
                                                   in sorted(item["hetero"].items()))
                  + "  (removed when the target is prepared)")
        if item["insertion_codes"]:
            print(f"       insertion codes : {', '.join(item['insertion_codes'])}")
        labels.append((f"{item['chain']}  -  {item['residue_count']} residues"
                       f"{'  ' + description if description else ''}", item["chain"]))

    differences = described["differences"]
    print(f"\nengineered differences this file declares: {len(differences)}")
    for difference in differences[:20]:
        print(f"  chain {difference['chain'] or '?'}  {difference['residue']}"
              f"{difference['number']} vs {difference['database']} {difference['accession']} "
              f"{difference['reference_residue']}{difference['reference_number']}  "
              f"{difference['note']}")
    if len(differences) > 20:
        print(f"  ... and {len(differences) - 20} more")
    if not differences:
        print("  (the file declares none; this is not proof the construct is wild type)")

    if entry:
        print("\nresidue-number mapping:")
        for item in described["chains"]:
            entity = hpc.entity_of_chain(entry, item["chain"])
            if entity:
                for line in hpc.numbering_mapping(entity):
                    print(f"  chain {item['chain']}: {line}")
                break

    if len(labels) == 1:
        chain.options = labels
        chain.value = labels[0][1]
        print(f"\nchain {labels[0][1]} is the only polymer chain, so it is pre-selected.")
    else:
        # A Dropdown selects its first option the moment options are assigned, so a
        # real chain must not be first: an empty sentinel goes there instead, and
        # 'prepare and add' refuses while it is selected. Otherwise the form would
        # pre-select a chain for the user, which in a complex is usually the antibody.
        chain.options = [("- choose the antigen chain -", "")] + labels
        chain.value = ""
        print("\nSeveral polymer chains: choose the antigen chain above. Nothing is "
              "pre-selected, because the longest chain is usually the antibody.")


def on_search(_=None):
    with lookup_out:
        lookup_out.clear_output(wait=True)
        found, error = hpc.rcsb_search(query.value, limit=10)
        if error:
            print(error)
            hits.options = []
            return
        print(f"RCSB entries for {query.value!r}, best match first: {', '.join(found)}")
        labels = []
        for identifier in found:
            report, entry_error = hpc.rcsb_entry_report(identifier)
            if entry_error:
                labels.append((identifier, identifier))
                print(f"  {identifier}  (could not read its summary: {entry_error})")
                continue
            resolution = f", {report['resolution']} A" if report.get("resolution") else ""
            summary = f"{identifier}  {report['title'][:70]}"
            print(f"  {summary}")
            print(f"       {', '.join(report['methods'])}{resolution}"
                  f"   released {report['released'][:10]}")
            for item in report["entities"]:
                print(f"       entity {item['entity_id']} chain {','.join(item['chains'])}: "
                      f"{item['description'][:60]}")
            labels.append((summary, identifier))
        hits.options = labels
        if labels:
            print("\nPick one above, then press 'fetch and inspect'.")


def on_inspect(_=None):
    with lookup_out:
        lookup_out.clear_output(wait=True)
        identifier = (hits.value if source.value.startswith("search") and hits.value
                      else pdb_id.value)
        report, error = hpc.rcsb_entry_report(identifier)
        if error:
            print(error)
            return
        LOCATED["entry"] = report
        resolution = f", {report['resolution']} A" if report.get("resolution") else ""
        print(f"RCSB {report['pdb_id']}  {report['title']}")
        print(f"method   : {', '.join(report['methods'])}{resolution}")
        print(f"released : {report['released'][:10]}")
        if report["citation"]:
            print(f"citation : {report['citation']}")
        for problem in report["problems"]:
            print(f"problem  : {problem}")
        path, error = hpc.rcsb_download(report["pdb_id"])
        if error:
            print(error)
            return
        print()
        _describe(path, note=f"downloaded to {path}")


def on_read_upload(_=None):
    with lookup_out:
        lookup_out.clear_output(wait=True)
        # ipywidgets 7 gives a dict keyed by filename, 8 a tuple of dicts.
        items = upload.value
        records = list(items.values()) if isinstance(items, dict) else list(items or ())
        names = list(items.keys()) if isinstance(items, dict) else [
            record.get("name", "upload") for record in records]
        if not records:
            print("choose a file first")
            return
        stored = hpc.save_upload(names[0], bytes(records[0]["content"]))
        for problem in stored["problems"]:
            print(f"problem  : {problem}")
        if not stored["output"]:
            return
        LOCATED.pop("entry", None)
        _describe(stored["output"], note=f"stored unchanged at {stored['output']}")


def on_read_path(_=None):
    with lookup_out:
        lookup_out.clear_output(wait=True)
        located = Path(file_path.value.strip()).expanduser()
        if not located.is_file():
            print(f"not a file on this machine: {located}")
            return
        LOCATED.pop("entry", None)
        _describe(located)


def show_targets():
    with chosen:
        chosen.clear_output(wait=True)
        if not TARGETS:
            print("no target yet")
            return
        print(f"{len(TARGETS)} target entr(ies) for the campaign JSON "
              "(several make it a multi-target campaign):")
        for index, target in enumerate(TARGETS, start=1):
            print(f"{index}. {json.dumps(target)}")


def _optional_int(box):
    text = box.value.strip()
    return int(text) if text.lstrip("-").isdigit() else None


def on_add(_=None):
    with lookup_out:
        try:
            if source.value == "a shipped preset":
                entry = hpc.preset_target(shipped.value)
                print(f"shipped preset {shipped.value}: {json.dumps(entry)}")
            elif source.value == "paste a sequence":
                written = hpc.write_fasta_target(name.value or "pasted_target", pasted.value)
                for problem in written["problems"]:
                    print(f"problem  : {problem}")
                if not written["output"]:
                    return
                entry = {"name": written["name"], "target_path": written["output"]}
                print(f"FASTA target written to {written['output']} "
                      f"({written['residue_count']} residues). BindCraft2 predicts a FASTA "
                      "target's own structure as part of the campaign.")
            else:
                if not LOCATED.get("path"):
                    print("read or fetch a file first")
                    return
                located = Path(LOCATED["path"])
                if hpc.structure_format(located) == "fasta":
                    entry = {"name": name.value.strip() or located.stem,
                             "target_path": str(located)}
                    print(f"using the FASTA file as it is: {located}")
                else:
                    if not chain.value:
                        print("choose the antigen chain first - this step does not guess")
                        return
                    provenance = []
                    report = LOCATED.get("entry")
                    if report:
                        provenance.append(f"RCSB entry {report['pdb_id']}: {report['title']}")
                        provenance.append(f"Method: {', '.join(report['methods'])}"
                                          + (f", {report['resolution']} A"
                                             if report.get("resolution") else ""))
                        entity = hpc.entity_of_chain(report, chain.value)
                        if entity:
                            provenance.append(f"Entity: {entity.get('description', '')}".rstrip())
                            provenance += hpc.numbering_mapping(entity)
                    prepared = hpc.prepare_target(
                        located, chain=chain.value, name=name.value.strip(),
                        first=_optional_int(first_residue), last=_optional_int(last_residue),
                        provenance=provenance)
                    for problem in prepared["problems"]:
                        print(f"problem  : {problem}")
                    if not prepared["output"]:
                        return
                    print(f"target written to {prepared['output']}")
                    print(f"  chain       : {prepared['chain']}  ({prepared['chain_note']})")
                    print(f"  residues    : {prepared['residue_count']}")
                    print(f"  segments    : {hpc.span_text(prepared['segments'])}")
                    print(f"  absent      : {hpc.span_text(prepared['absent'])}")
                    print(f"  sequence    : {prepared['sequence']}")
                    for item, count in sorted(prepared["removed"].items()):
                        print(f"  removed     : {item} ({count} atoms)")
                    print("  numbering   : the source file's author numbering, unchanged - "
                          "give hotspots in it")
                    entry = {"name": prepared["name"], "target_path": prepared["output"],
                             "chains": prepared["chain"]}

            for key, box in (("hotspots", hotspots), ("coldspots", coldspots)):
                if box.value.strip():
                    entry[key] = box.value.strip()
            if not entry.get("hotspots"):
                print("  hotspots    : none given, which is allowed - BindCraft2 chooses its "
                      "own interface on the target")
            elif geometry.value and entry.get("chains"):
                numbers = [int(found) for found in
                           __import__("re").findall(r"\d+", entry["hotspots"])]
                print("\n  geometry of the hotspots on this file:")
                for line in hpc.geometry_notes(entry["target_path"], entry["chains"].split(",")[0],
                                               numbers):
                    print(f"    - {line}")
            TARGETS.append(entry)
            print(f"\nadded: {json.dumps(entry)}")
        except Exception as failure:
            print(f"not added: {type(failure).__name__}: {failure}")
            return
    show_targets()


def on_clear(_=None):
    TARGETS.clear()
    LOCATED.clear()
    show_targets()


source.observe(on_source, names="value")
run_search.on_click(on_search)
inspect.on_click(on_inspect)
read_upload.on_click(on_read_upload)
read_path.on_click(on_read_path)
add.on_click(on_add)
clear.on_click(on_clear)
on_source()
display(widgets.VBox([source, query, run_search, hits, pdb_id, inspect, upload, read_upload,
                      pasted, file_path, read_path, shipped,
                      widgets.HTML("<hr><b>Selection</b>"), chain, name,
                      first_residue, last_residue,
                      widgets.HTML("<b>Optional steering</b>"), hotspots, coldspots, geometry,
                      widgets.HBox([add, clear]), lookup_out, chosen]))
show_targets()


## 3 - Binder and campaign settings

Ten binder formats, all of them upstream's own. `max_trajectories` is the work budget;
`number_of_final_designs` is when the campaign stops early because it has enough. A short
campaign can legitimately accept nothing.

Scaffolded formats (VHH, ARP, scFv, Fab) bring their own domain lengths from the scaffold they
design on, so the length boxes disappear for them - that is correct, not a missing control.

The form opens on **scFv** as the platform's default example format. Change it freely: a de novo
binder, a peptide, a VHH or a Fab against the same target is one dropdown away, and the settings
file is written for you either way.

Every acceptance filter, loss and CDR span comes from the shipped presets and stays enabled.


In [ ]:
campaign_name = widgets.Text(value="cd19_scfv_discovery", description="campaign name",
                             style=STYLE, layout=wide())
binder_type = widgets.Dropdown(options=list(hpc.BINDER_TYPES), value="scFv",
                               description="binder type", style=STYLE, layout=wide())
shortest = widgets.BoundedIntText(value=60, min=1, max=2000, description="shortest binder",
                                  style=STYLE, layout=wide("340px"))
longest = widgets.BoundedIntText(value=100, min=1, max=2000, description="longest binder",
                                 style=STYLE, layout=wide("340px"))
copies = widgets.BoundedIntText(value=1, min=1, max=12, description="copies (oligomer)",
                                style=STYLE, layout=wide("340px"))
properties = widgets.SelectMultiple(options=list(hpc.PROPERTY_OPTIONS), rows=8,
                                    description="extra properties", style=STYLE, layout=wide())
final_designs = widgets.BoundedIntText(value=10, min=1, max=500, description="final designs wanted",
                                       style=STYLE, layout=wide("340px"))
trajectories = widgets.BoundedIntText(value=500, min=1, max=100000, description="max trajectories",
                                      style=STYLE, layout=wide("340px"))
workers = widgets.BoundedIntText(value=1, min=1, max=8, description="workers per GPU",
                                 style=STYLE, layout=wide("340px"))
resume = widgets.Checkbox(value=True, indent=False,
                          description="resume into the same folder if resubmitted")
write = widgets.Button(description="write settings file", button_style="primary",
                       layout=widgets.Layout(width="200px"))
written = widgets.Output()
CONFIG_PATH = None
SETTINGS = None

def on_type(change=None):
    modality = hpc.BINDER_TYPES[binder_type.value]
    scaffolded = modality in hpc.SCAFFOLDED_TYPES
    for box in (shortest, longest):
        box.layout.display = "none" if scaffolded else None
    copies.layout.display = None if modality == "homo_oligomer" else "none"
    preset = hpc.modality_lengths(modality)
    if preset and not scaffolded:
        shortest.value, longest.value = int(preset[0]), int(preset[1])

def on_write(_=None):
    global CONFIG_PATH, SETTINGS
    with written:
        written.clear_output(wait=True)
        if not TARGETS:
            print("add a target in cell 2 first")
            return
        try:
            SETTINGS = hpc.build_campaign_settings(
                campaign_name=campaign_name.value,
                targets=TARGETS,
                modality=hpc.BINDER_TYPES[binder_type.value],
                properties=[hpc.PROPERTY_OPTIONS[label] for label in properties.value],
                binder_lengths=(shortest.value, longest.value),
                number_of_final_designs=final_designs.value,
                max_trajectories=trajectories.value,
                workers_per_gpu=workers.value,
                copies=copies.value,
                extra={"resume": True} if resume.value else None)
            CONFIG_PATH = hpc.write_campaign_config(SETTINGS)
        except Exception as failure:
            print(f"not written: {type(failure).__name__}: {failure}")
            return
        print(f"deployment      : {hpc.DEPLOYMENT}")
        print(f"settings written: {CONFIG_PATH}")
        print(f"results will go : {SETTINGS['project_folder']}\n")
        print(json.dumps(SETTINGS, indent=2))
        print("\nAcceptance filters come from the presets above and are left enabled.")
        if hpc.BINDER_TYPES[binder_type.value] in hpc.SCAFFOLDED_TYPES:
            print(f"The {binder_type.value} scaffold decides the domain lengths; the length "
                  "boxes above are ignored for it.")

binder_type.observe(on_type, names="value")
write.on_click(on_write)
on_type()
display(widgets.VBox([campaign_name, binder_type, shortest, longest, copies, properties,
                      final_designs, trajectories, workers, resume, write, written]))


## 4 - Submit the campaign to an H200 through SLURM

This calls `sbatch lab/slurm/run_bindcraft_h200.slurm <settings.json>`: partition `gpu`,
`--gres=gpu:h200:1`, one task, 8 CPUs, 64 GB host RAM, 08:00:00, pinned to `cmri-gpu05` - the
proven configuration, verified by job 1015050 (COMPLETED, exit 0:0, JAX 0.11.2 `backend=gpu` on
the H200 NVL). BindCraft2 is executed directly by the batch script; nothing runs under `srun`,
which fails on this cluster with SLURM GRES errors.

The launcher refuses to start if `bindcraft` is imported from outside this source copy or if JAX
cannot see a GPU, and it never downloads weights. The job id is recorded under the results root,
so a later notebook session finds it again.

Leave *node* empty to use the pinned default; set it to another H200 node to override it.

In [ ]:
account = widgets.Text(description="SLURM account (if any)", placeholder="leave empty unless your site requires one",
                       style=STYLE, layout=wide("720px"))
af2 = widgets.Text(description="AlphaFold params dir",
                   value=str(hpc.alphafold_parameters()[0] or ""),
                   placeholder="directory holding params_<model>.npz", style=STYLE, layout=wide("720px"))
node = widgets.Text(description="node (optional)", value="",
                    placeholder="empty = the launcher's pinned cmri-gpu05",
                    style=STYLE, layout=wide("720px"))
dry_run = widgets.Checkbox(value=False, description="dry run (print the command, submit nothing)",
                           indent=False)
submit = widgets.Button(description="submit to H200", button_style="danger",
                        layout=widgets.Layout(width="200px"))
submitted = widgets.Output()
SUBMISSION = None

def on_submit(_=None):
    global SUBMISSION
    with submitted:
        submitted.clear_output(wait=True)
        if CONFIG_PATH is None:
            print("write a settings file in cell 3 first")
            return
        SUBMISSION = hpc.submit_campaign(CONFIG_PATH, account=account.value,
                                        af2_parameters=af2.value.strip(),
                                        nodelist=node.value.strip(),
                                        dry_run=dry_run.value)
        print("command :", " ".join(SUBMISSION.command))
        print("message :", SUBMISSION.message)
        if SUBMISSION.job_id:
            print("job id  :", SUBMISSION.job_id)
            print("stdout  :", SUBMISSION.log_out)
            print("stderr  :", SUBMISSION.log_err)
            print("results :", SUBMISSION.project_folder)
            print("\nThe campaign now runs in the batch job. Closing this notebook does not stop it.")
        else:
            print("\nNothing was submitted. Nothing here claims a GPU ran.")

submit.on_click(on_submit)
display(widgets.VBox([account, af2, node, dry_run, submit, submitted]))

## 5 - Job status and log

`squeue` while the job is queued or running, `sacct` once it has finished. The log is the only
place that proves the campaign ran on a GPU: look for `xulab-bc2: jax ... backend=gpu` and
`xulab-bc2: device gpu:0 NVIDIA H200`.

Every job also prints a GPU report before it designs anything: `xulab-diag:` lines for the SLURM
bindings, the `/dev/nvidia*` nodes and which `libcuda` would be loaded, then `xulab-gpu:` lines
for `cuInit` and the JAX backend, ending in `xulab-gpu: verdict`. If a job stops at the GPU gate,
read those lines first - they name the cause (a stub `libcuda`, a missing `/dev/nvidia-uvm`, or a
`CUDA_VISIBLE_DEVICES` the job was not given) instead of leaving you to guess.

In [ ]:
job_box = widgets.Text(description="job id (optional)", placeholder="leave empty for all recorded jobs",
                       style=STYLE, layout=wide())
lines = widgets.BoundedIntText(value=40, min=5, max=500, description="log lines", style=STYLE,
                               layout=wide("320px"))
refresh = widgets.Button(description="refresh status", button_style="primary",
                         layout=widgets.Layout(width="170px"))
show_log = widgets.Button(description="show log tail", layout=widgets.Layout(width="170px"))
cancel = widgets.Button(description="cancel this job", button_style="warning",
                        layout=widgets.Layout(width="170px"))
state_out = widgets.Output()

def current_job():
    typed = job_box.value.strip()
    if typed:
        return typed
    if SUBMISSION is not None and SUBMISSION.job_id:
        return SUBMISSION.job_id
    recorded = hpc.read_registry()
    return recorded[-1]["job_id"] if recorded else ""

def on_refresh(_=None):
    with state_out:
        state_out.clear_output(wait=True)
        records = hpc.tracked_jobs()
        if not records:
            print(f"no job recorded in {hpc.JOB_REGISTRY}")
            return
        for record in records:
            status = record.get("status", {})
            print(f"{record.get('job_id','?'):>10}  {record.get('campaign','?'):<28} "
                  f"{status.get('state','?'):<12} {status.get('elapsed','')} "
                  f"{status.get('exit_code', status.get('reason',''))}")
            print(f"            {record.get('project_folder','')}")

def on_log(_=None):
    with state_out:
        state_out.clear_output(wait=True)
        print(hpc.log_tail(current_job(), lines=lines.value))

def on_cancel(_=None):
    with state_out:
        state_out.clear_output(wait=True)
        job = current_job()
        print(hpc.cancel_job(job) if job else "no job id to cancel")

refresh.on_click(on_refresh)
show_log.on_click(on_log)
cancel.on_click(on_cancel)
display(widgets.VBox([job_box, lines, widgets.HBox([refresh, show_log, cancel]), state_out]))
on_refresh()

## 6 - Load a campaign and read its results

Works on a finished or a still-running campaign. With nothing accepted yet it shows the
candidates and the filters that rejected them, then the attempts and where they stopped - the
order the upstream output guide recommends.

In [ ]:
import pandas as pd   # tables only; the helper layer works without it

folders = hpc.campaign_folders()
picker = widgets.Dropdown(options=[folder.name for folder in folders] or ["(none found)"],
                          description="campaign", style=STYLE, layout=wide())
elsewhere = widgets.Text(description="or a full path", placeholder="/project/yli4/ypan/bindcraft/results/<name>",
                         style=STYLE, layout=wide("720px"))
rows = widgets.BoundedIntText(value=20, min=1, max=500, description="rows to show", style=STYLE,
                              layout=wide("320px"))
load = widgets.Button(description="load results", button_style="primary",
                      layout=widgets.Layout(width="170px"))
results_out = widgets.Output()
CAMPAIGN = None

def on_load(_=None):
    global CAMPAIGN
    with results_out:
        results_out.clear_output(wait=True)
        chosen_path = (Path(elsewhere.value.strip()) if elsewhere.value.strip()
                       else hpc.RESULTS_ROOT / picker.value)
        try:
            CAMPAIGN = hpc.load_campaign(chosen_path)
        except Exception as failure:
            print(f"{type(failure).__name__}: {failure}")
            return
        print(CAMPAIGN.summary())
        if CAMPAIGN.ranked:
            print("\nAccepted designs, best first:")
            display(hpc.ranked_view(CAMPAIGN, rows.value))
        if CAMPAIGN.candidates:
            print("\nScored candidates:")
            display(hpc.candidate_view(CAMPAIGN, rows.value))
            counts = hpc.rejection_counts(CAMPAIGN)
            if counts:
                print("rejected most often by:")
                for reason, count in list(counts.items())[:10]:
                    print(f"  {count:>5}  {reason}")
                print("('not measured' is a missing reading, not a low score)")
        if CAMPAIGN.attempts:
            print("\nTrajectory attempts (blank 'terminated' means gradient design completed):")
            display(hpc.attempt_view(CAMPAIGN, rows.value))
        if CAMPAIGN.metadata:
            keys = list(CAMPAIGN.metadata)[:10]
            print(f"\ncampaign_metadata.json keys: {keys}")

load.on_click(on_load)
display(widgets.VBox([picker, elsewhere, rows, load, results_out]))

## 7 - Top 10 accepted candidates, sequences and ranking

Up to ten **accepted** designs, in the order `3_Ranked/!_Ranked.csv` already holds
them - the pipeline's own `i_pDAE` ranking. If fewer passed, fewer are shown: a rejected
candidate is never promoted to fill a place.

For an antibody-format campaign (`scFv`, `Fab`, `VHH`) each design also gets its variable
domains named and its loops annotated. The domain is called VH or VL only when its framework
matches that chain of the scaffold the campaign designed on, and a loop is reported only when
the framework anchors on both sides of it are found in that design. Anything that cannot be
established that way prints as unavailable. The convention is printed with the table; the short
version is that loop spans come from the shipped modality preset's `mutate_positions`
declaration, and the quotable **HCDR3 length is the Kabat one**, measured between the conserved
Cys stem and the W-G-x-G motif located in the design itself.

A de novo binder campaign (the PD-L1 path) shows the same sequence, confidence and interface
readings here as it always did, with no antibody annotation.

The rejected candidates and the filters that rejected them stay available at the bottom of the
cell, and in full in cell 6.


In [ ]:
top_count = widgets.BoundedIntText(value=10, min=1, max=50, description="designs to show",
                                   style=STYLE, layout=wide("340px"))
modality_override = widgets.Text(description="modality (optional)", placeholder="read from the campaign; e.g. scFv to force it",
                                 style=STYLE, layout=wide("720px"))
show_convention = widgets.Checkbox(value=False, description="print the numbering convention in full",
                                   indent=False)
read_top = widgets.Button(description="show top candidates", button_style="primary",
                          layout=widgets.Layout(width="220px"))
top_out = widgets.Output()
REPORT = None


def _print_domain(domain):
    label = domain.get("domain", "?")
    print(f"     {label}  ({domain.get('length', 0)} aa, output chain "
          f"{domain.get('chain_position', '?')})")
    if domain.get("anchors_matched"):
        print(f"       identified by : {domain['anchors_matched']}")
    print(f"       sequence      : {domain.get('sequence') or '(unavailable)'}")
    for note in domain.get("notes", []):
        print(f"       note          : {note}")
    paratope = domain.get("paratope") or {}
    for loop in domain.get("loops", []):
        contacts = (paratope.get("loops") or {}).get(loop["name"], [])
        if loop.get("sequence") is None:
            print(f"       {loop['name']:<6} {loop['scaffold_span']:<10} unavailable: "
                  f"{loop['unavailable']}")
            continue
        print(f"       {loop['name']:<6} {loop['scaffold_span']:<10} {loop['sequence']}"
              f"  ({loop['length']} aa)"
              + (f"  contacts the target at {','.join(contacts)}" if contacts else ""))
    if paratope.get("framework"):
        print(f"       framework contacts : {','.join(paratope['framework'])}")
    if paratope.get("unmapped"):
        print(f"       unmapped contacts  : {','.join(paratope['unmapped'])} "
              "(residue label did not match this sequence, so it was not placed in a loop)")


def on_top(_=None):
    global REPORT
    with top_out:
        top_out.clear_output(wait=True)
        if CAMPAIGN is None:
            print("load a campaign in cell 6 first")
            return
        REPORT = hpc.top_designs(CAMPAIGN, limit=top_count.value,
                                 modality=modality_override.value.strip())
        print(f"campaign : {CAMPAIGN.folder}")
        print(f"modality : {REPORT['modality'] or 'not an antibody format; no loop annotation'}")
        print(f"accepted : {len(REPORT['designs'])} shown of {CAMPAIGN.accepted_count} in "
              f"3_Ranked/!_Ranked.csv")
        for note in REPORT["notes"]:
            print(f"note     : {note}")
        if not REPORT["designs"]:
            print("\nNothing accepted yet, so there is no candidate list. Cell 6 shows the")
            print("candidates and the filters that rejected them.")
            return
        if show_convention.value and REPORT["convention"]:
            print("\n" + REPORT["convention"])

        for entry in REPORT["designs"]:
            print("\n" + "=" * 86)
            print(f"rank {entry['rank']:<4} {entry['design']}")
            print(f"     structure : {entry['structure'] or '(no complex file found)'}")
            if entry["confidence"]:
                print("     confidence: " + "  ".join(f"{key}={value}" for key, value
                                                      in entry["confidence"].items()))
            for key, value in entry["interface"].items():
                print(f"     {key:<34}{value}")
            print(f"     predicted target contacts : {entry['target_residues'] or '(not recorded)'}")
            if entry.get("hcdr3"):
                hcdr3 = entry["hcdr3"]
                if hcdr3.get("sequence"):
                    print(f"     HCDR3 (Kabat) : {hcdr3['sequence']}  ({hcdr3['length']} aa, "
                          f"stem {hcdr3['stem']} .. motif {hcdr3['motif']})")
                else:
                    print(f"     HCDR3 (Kabat) : unavailable: {hcdr3.get('unavailable', '')}")
            for domain in entry["domains"]:
                _print_domain(domain)
        print("\n" + "=" * 86)
        print("i_pDAE / i_pTM / i_pAE / pLDDT / pTM are prediction confidence. They are not")
        print("affinity, not specificity, not stability and not probabilities of binding.")
        print("Buried area is binder-side, in square angstrom; i_pAE is divided by 31 A.")

        columns, rows = hpc.cdr_table(CAMPAIGN, limit=top_count.value,
                                      modality=modality_override.value.strip())
        if rows:
            print("\nAnnotated loop table (one row per loop; empty sequence means unavailable):")
            display(hpc.as_frame(columns, rows, columns))

        if CAMPAIGN.candidates:
            print("\nRejected candidates and the filters that rejected them "
                  f"({len(CAMPAIGN.candidates)} rows in 2_Refolded/!_Refolded.csv):")
            display(hpc.candidate_view(CAMPAIGN, 20))


read_top.on_click(on_top)
display(widgets.VBox([top_count, modality_override, show_convention, read_top, top_out]))


## 8 - 3D review of the accepted complexes, one at a time

**Individual View is the default.** The dropdown is built from the accepted designs that
actually have a complex file, labelled by rank and design ID, and shows **one** complex at a
time; Previous and Next walk the ranking. No design is ever listed twice under two ranks, and a
design whose file is missing is left out of the dropdown rather than silently standing in for
another.

`colour by chain` separates the target from the binder chains - for an scFv campaign that is
the target, VH and VL, and the chain list under the viewer says which chain is which wherever the
scaffold framework proves it. `colour by pLDDT` colours by local prediction confidence
(BindCraft2 writes per-residue pLDDT into the B-factor column, 0-100).

The optional highlight draws sticks on the Kabat HCDR3 or on the contacted target residues
recorded for that design. A highlight that cannot be established from the recorded metadata is
skipped and says so, rather than being drawn in the wrong place.

**Compare View** is still here, now up to **10** complexes in a grid.

`py3Dmol` needs internet access for 3Dmol.js. On a closed network this cell says so, and the
tables plus the CIF downloads in cell 9 remain the way to inspect the complexes.


In [ ]:
mode = widgets.ToggleButtons(options=["Individual View", "Compare View"],
                             value="Individual View", description="view",
                             style={"description_width": "180px"})
design_picker = widgets.Dropdown(options=["(load a campaign first)"], description="design",
                                 style=STYLE, layout=wide("560px"))
previous_design = widgets.Button(description="< Previous", layout=widgets.Layout(width="130px"))
next_design = widgets.Button(description="Next >", layout=widgets.Layout(width="130px"))
colour_by = widgets.Dropdown(options=["chain", "pLDDT"], description="colour by", style=STYLE,
                             layout=wide("340px"))
highlight = widgets.Dropdown(options=["none", "HCDR3", "epitope"], value="none",
                             description="highlight", style=STYLE, layout=wide("340px"))
how_many = widgets.BoundedIntText(value=2, min=1, max=10, description="how many (compare)",
                                  style=STYLE, layout=wide("340px"))
refresh = widgets.Button(description="refresh design list", layout=widgets.Layout(width="190px"))
draw = widgets.Button(description="show structure", button_style="primary",
                      layout=widgets.Layout(width="180px"))
view_out = widgets.Output()
AVAILABLE = []


def on_mode(change=None):
    individual = mode.value == "Individual View"
    for box in (design_picker, previous_design, next_design, highlight):
        box.layout.display = None if individual else "none"
    how_many.layout.display = "none" if individual else None


def on_refresh(_=None):
    global AVAILABLE
    with view_out:
        view_out.clear_output(wait=True)
        if CAMPAIGN is None:
            print("load a campaign in cell 6 first")
            return
        AVAILABLE = hpc.accepted_designs(CAMPAIGN, limit=10)
        if not AVAILABLE:
            design_picker.options = ["(no accepted complex file yet)"]
            print(f"no accepted complex files in {CAMPAIGN.folder / '3_Ranked'}.")
            print("Cell 6 shows whether anything has been accepted at all.")
            return
        design_picker.options = [(entry["label"], index)
                                 for index, entry in enumerate(AVAILABLE)]
        design_picker.value = 0
        print(f"{len(AVAILABLE)} accepted design(s) with a complex file, best first:")
        for entry in AVAILABLE:
            print(f"  {entry['label']}   {entry['path'].name}")


def _step(offset):
    if not AVAILABLE or not isinstance(design_picker.value, int):
        return
    design_picker.value = (design_picker.value + offset) % len(AVAILABLE)
    on_draw()


def on_draw(_=None):
    with view_out:
        view_out.clear_output(wait=True)
        if CAMPAIGN is None:
            print("load a campaign in cell 6 first")
            return
        if mode.value == "Compare View":
            try:
                view, shown = hpc.show_structures(CAMPAIGN, how_many.value, colour_by.value)
            except RuntimeError as failure:
                print(failure)
                return
            for index, path in enumerate(shown, start=1):
                print(f"{index}. {path.name}  chains {','.join(hpc.chain_names(path)) or '?'}")
            view.show()
            return

        if not AVAILABLE or not isinstance(design_picker.value, int):
            print("press 'refresh design list' first")
            return
        try:
            view, entry, chains = hpc.show_structure(CAMPAIGN, design_picker.value,
                                                     colour_by.value, highlight=highlight.value)
        except RuntimeError as failure:
            print(failure)
            return

        row = next((item for item in CAMPAIGN.ranked
                    if (item.get("design") or "").strip() == entry["design"]), {})
        print(f"rank      : {entry['rank']} of {len(AVAILABLE)} accepted design(s) shown here")
        print(f"design ID : {entry['design']}")
        print(f"file      : {entry['path'].name}")
        print(f"full path : {entry['path']}")
        readings = "  ".join(f"{key}={row[key]}" for key in hpc.CONFIDENCE_COLUMNS
                             if key in row and str(row[key]).strip())
        print(f"confidence: {readings or '(not recorded)'}")
        for key in ("Interface_Residues", "Interface_BuriedArea", "Backbone_Clashes",
                    "All_Atom_Clashes", "Interface_Target_Residues"):
            if str(row.get(key, "")).strip():
                print(f"{key:<26}{row[key]}")
        print("chains    :")
        for chain in chains:
            print(f"  {chain['chain']}  {chain['length']:>4} residues  {chain['domain']}")
            if chain.get("highlight"):
                print(f"      highlight: {chain['highlight']}")
        print(f"colour by : {colour_by.value}"
              + ("  (per-residue pLDDT, 0-100, from the B-factor column)"
                 if colour_by.value == "pLDDT" else "  (one colour per chain)"))
        view.show()


mode.observe(on_mode, names="value")
design_picker.observe(lambda change: None, names="value")
previous_design.on_click(lambda _=None: _step(-1))
next_design.on_click(lambda _=None: _step(1))
refresh.on_click(on_refresh)
draw.on_click(on_draw)
on_mode()
display(widgets.VBox([mode, widgets.HBox([refresh, draw]), design_picker,
                      widgets.HBox([previous_design, next_design]),
                      colour_by, highlight, how_many, view_out]))


## 9 - Export and download

Collects the binder FASTA, the ranked / candidate / attempt CSVs, `summary.csv`,
`campaign_metadata.json` and the accepted `.cif` complexes into one folder under
`results/exports/<campaign>/`, plus a zip. Nothing is written into the source checkout, so a
source update cannot delete an export.

An antibody-format campaign additionally gets `<campaign>_VH_VL.fasta`, with one record per
variable domain named `<design>_VH` / `<design>_VL`, `<campaign>_CDRs.csv` with one row per
annotated loop, and `<campaign>_CDR_numbering.txt` recording the convention those spans were
read with, so the table can be checked later. A de novo binder export is unchanged.


In [ ]:
limit = widgets.BoundedIntText(value=20, min=1, max=500, description="designs to export",
                               style=STYLE, layout=wide("320px"))
run_export = widgets.Button(description="export results", button_style="primary",
                            layout=widgets.Layout(width="180px"))
export_out = widgets.Output()
EXPORT = None

def on_export(_=None):
    global EXPORT
    with export_out:
        export_out.clear_output(wait=True)
        if CAMPAIGN is None:
            print("load a campaign in cell 6 first")
            return
        EXPORT = hpc.export_campaign(CAMPAIGN, limit=limit.value)
        print(f"folder    : {EXPORT['folder']}")
        print(f"files     : {len(EXPORT['files'])}")
        print(f"sequences : {EXPORT['sequences']} FASTA record(s)")
        if EXPORT.get("zip"):
            print(f"zip       : {EXPORT['zip']}  ({EXPORT.get('zip_bytes', 0) / 1e6:.1f} MB)")
        for name in sorted(set(EXPORT["files"]))[:40]:
            print("   ", name)
        print("\nFASTA preview:")
        print(hpc.binder_fasta(CAMPAIGN, limit=3) or "(nothing accepted yet)")
        print("Download from the Jupyter file browser, or:")
        print(f"  scp <user>@<hpc-host>:{EXPORT.get('zip') or EXPORT['folder']} .")
        hpc.download_links([EXPORT.get("zip")] if EXPORT.get("zip") else [])

run_export.on_click(on_export)
display(widgets.VBox([limit, run_export, export_out]))

## 10 - Audit the campaign

Runs `lab/scripts/validate_outputs.py`, the same read-only validator used outside the notebook.
It grants **Level A (compute smoke)** only on explicit JAX GPU backend evidence plus a clean
campaign exit, and **Level B (end-to-end)** only when every accepted design has a sequence and a
structure. Zero accepted designs is reported as NOT ACHIEVED, which is not a failure.

In [ ]:
log_pattern = widgets.Text(description="log pattern",
                           value=str(hpc.LOGS_ROOT / "xulab_bc2_*.out"),
                           style=STYLE, layout=wide("720px"))
audit = widgets.Button(description="run validator", button_style="primary",
                       layout=widgets.Layout(width="180px"))
audit_out = widgets.Output()

def on_audit(_=None):
    with audit_out:
        audit_out.clear_output(wait=True)
        if CAMPAIGN is None:
            print("load a campaign in cell 6 first")
            return
        report_file = Path(EXPORT["folder"]) / "validation_report.md" if EXPORT else ""
        status, report = hpc.validate_campaign(CAMPAIGN.folder, log_glob=log_pattern.value,
                                               report_path=str(report_file) if report_file else "")
        print(report)
        print(f"validator exit status {status} "
              f"(0 = nothing failed, 1 = at least one FAIL, 2 = folder unreadable)")
        if report_file:
            print(f"report written to {report_file}")

audit.on_click(on_audit)
display(widgets.VBox([log_pattern, audit, audit_out]))

---

### Reading the outcome honestly

* **Zero accepted designs** after a short campaign is an ordinary result. Raise
  `max_trajectories` and resubmit; the campaign resumes into the same folder.
* **An accepted design is a computational result.** `i_pDAE`, `i_pTM` and pLDDT are prediction
  confidence, not affinity, specificity or stability. Nothing in this notebook is experimental
  evidence of binding.
* **A predicted target contact is a prediction.** The contacted target residues in step 7 and
  the highlight in step 8 are read from the complex the pipeline predicted. They are not a
  mapped epitope.
* **GPU use is proven only by the job log.** This kernel never asserts that an H200 ran.
* **Source and data are separate trees.** The source copy lives in `$HOME`, everything a
  campaign writes lives under the project root, and both are printed by cell 1. Replacing the
  source cannot delete a campaign, and results never consume the HOME quota.
* **Which deployment you are on is printed by cell 1.** A development deployment has its own
  source tree, project root, targets, results, logs and job registry; nothing it submits can
  write into the production tree.

### Starting a campaign from scratch

```bash
# once, on a login node, for whichever deployment you are setting up
mkdir -p "$XULAB_PROJECT_ROOT"/{results,logs,cache/jax,tmp,targets}
cd "$XULAB_BC2_HOME"
bash lab/scripts/hpc_preflight.sh          # must print RESULT: READY

# then, in Jupyter
#   open lab/notebooks/xulab_bindcraft_h200.ipynb with the bindcraft2 kernel
#   cell 1 -> cell 2 (search a target, pick its chain) -> cell 3 (format and budget)
#   -> cell 4 submit -> cell 5 to watch -> cells 6-10 for results
```

A target can also be prepared outside the notebook, which is the route to use when the kernel
has no internet access:

```bash
python lab/scripts/prepare_target.py --search "CD19"
python lab/scripts/prepare_target.py --entry 6AL5 --describe
python lab/scripts/prepare_target.py --entry 6AL5 --chain A --first 21 --last 277 \
    --name hCD19_ECD --contacts
```

Upstream documentation: `docs/source/installation.md`, `docs/source/reference.md`,
`docs/source/outputs.md`. Xu Lab deployment notes: `README_XULAB.md`, `lab/README.md`.
